In [ ]:
from math import *
from build123d import *
from ocp_vscode import *
from cad_utils import load_pcb, fast_copy, copy_located, export
from knurled_cylinder import KnurledCylinder

set_port(3939)
set_viewer_config(port=3939)

In [ ]:
radius = 10
height = 16
knurl_height = 13
knurls = 14
knurl_depth = 2
wall_thickness = 1.5
outer_wall_thickness = 2.5
divot_depth = 1

# shaft is a D shape
shaft_radius = 3
shaft_cut_offset = 1.5
shaft_depth = 10
shaft_tightness = 0.2
shaft_peg_width = wall_thickness


In [ ]:
reset_show()

with BuildPart() as encoder_knob_outer:
  KnurledCylinder(radius, knurl_height, knurls, knurl_depth, flatten_top_knurls=True, align=(Align.CENTER, Align.CENTER, Align.MIN))
  # Cylinder(radius-1, knurl_height, align=(Align.CENTER, Align.CENTER, Align.MIN))

  bottom: Face = faces().sort_by(Axis.Z)[-1]

  top_radius = radius - (height - knurl_height)
  top_knurl_radius = edges().sort_by(Axis.Z).last.radius

  with BuildSketch(Plane((0, 0, knurl_height), (0, 1, 0), (1, 0, 0))) as s:
    with BuildLine() as l:
      horiz = Line((0,0), (top_knurl_radius, 0))
      center = Line((0,0), (0, height-knurl_height-divot_depth))
      h = PolarLine(horiz@1, height-knurl_height, 90+45, length_mode=LengthMode.VERTICAL, mode=Mode.PRIVATE)
      c = Vector((h@1).X/2,height-knurl_height-divot_depth)
      p = [
        horiz@1,
        h@0.5,
        (h@1) + Vector(divot_depth/2, 0),
        (h@1),
        (h@1) - Vector(divot_depth/2, 0),
        c,
        center@1
      ]
      Bezier(p)
    make_face()
  revolve(s.sketch)

show_object(encoder_knob_outer.part)

In [ ]:
reset_show()

def make_pegs(shaft, bottom=False):
  with BuildSketch(mode=Mode.PRIVATE) as s:
    Circle(radius - outer_wall_thickness)

    circle_edge = shaft.edges().filter_by(GeomType.CIRCLE).first
    line_edge = shaft.edges().filter_by(GeomType.LINE).first
    split_edges = [line_edge] + [circle_edge.trim(i / 3, (i + 1) / 3) for i in range(3)]
    split_edges = [edge.trim(0.1, 0.9) for edge in split_edges]
    split_edges[0] = split_edges[0].reversed()

    with BuildSketch(mode=Mode.PRIVATE) as peg_line:
      Rectangle(shaft_peg_width, radius, align=(Align.CENTER, Align.MIN))

    with BuildSketch(mode=Mode.SUBTRACT) as pegs:
      for edge in split_edges:
        q = make_face(offset(edge, amount=shaft_peg_width, side=Side.RIGHT, mode=Mode.PRIVATE))
        peg_angle = q.center().get_signed_angle(Vector(0, 1))
        add(peg_line.face().rotate(Axis.Z, peg_angle).translate(q.center()))
        if bottom:
          make_face(offset(edge, amount=-shaft_tightness, side=Side.RIGHT, mode=Mode.PRIVATE), mode=Mode.SUBTRACT)
    fillet(vertices(), (shaft_peg_width /2) - shaft_tightness)
  return s.sketch

with BuildPart() as encoder_knob_inner:
  with BuildSketch(mode=Mode.PRIVATE) as shaft:
    Circle(shaft_radius)
    with Locations((0, shaft_cut_offset)):
      Rectangle(shaft_radius * 2, shaft_radius * 2, align=(Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)

  bottom = make_pegs(shaft, bottom=True)
  top = make_pegs(shaft).translate((0, 0, shaft_depth))

  show_object(bottom)
  show_object(top)

  loft([bottom, top])

show_object(encoder_knob_inner.part)


In [ ]:
reset_show()

with BuildPart() as encoder_knob:
  add(encoder_knob_outer.part)
  add(encoder_knob_inner.part, mode=Mode.SUBTRACT)

encoder_knob = encoder_knob.part
encoder_knob.label = "encoder_knob"

show_object(encoder_knob)

In [ ]:
export(encoder_knob)